# AIaaS — Industry-Comparable LLM Serving Benchmark (vLLM)

This notebook produces **serving numbers you can compare against public results**
(vendor blogs, community runs, the MLPerf-style methodology) because it uses the
*same harness everyone uses*: a real **vLLM OpenAI server** driven by vLLM's
**`bench serve`** load generator over the **ShareGPT** request distribution.

It reports the metrics the industry reports:
- **TTFT** (time-to-first-token) — mean / median / p99
- **TPOT / ITL** (time per output token / inter-token latency)
- **Output throughput** (tokens/s) and **request throughput** (req/s)
- A **request-rate sweep** so you get the latency-vs-throughput curve (the "knee")

> **Why this is different from the PoC proxy notebook:** the PoC uses raw
> `transformers` eager generation, a single prompt, and no load generator — it
> measures the *framework*, not the *hardware ceiling*, and isn't reproducible by
> others. This notebook fixes all three: optimized runtime (vLLM), a real dataset
> distribution, and a standardized load generator.

### Requirements
- A **CUDA GPU** runtime (A100 / Colab T4·L4·A100 / etc.). Will not run on CPU.
- ~10–20 min including model download.

### Portability note
vLLM pins its own CUDA-matched `torch`. Installing it **may upgrade torch and
require a runtime restart** (Colab: *Runtime → Restart*, then run from cell 2).
On locked-down environments (SageMaker Studio Lab, Kaggle) vLLM may not install —
that is expected; use the PoC proxy notebook there instead.


## 1. Install vLLM

In [ ]:
import subprocess, sys

# vLLM brings a compatible torch; do NOT pre-pin torch yourself.
# The [bench] extra is required for the `vllm bench serve` load generator used below.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "vllm[bench]", "pandas"], check=False)

try:
    import vllm
    print("vLLM", vllm.__version__)
except Exception as e:
    print("vLLM not importable yet:", e)
    print("If torch was upgraded, RESTART the runtime and re-run from this cell.")


## 2. Environment & hardware capture
Every result is tagged with this so on-prem vs Colab is comparable.

In [ ]:
import os, json, platform, subprocess, datetime
import torch

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

CUDA = torch.cuda.is_available()
assert CUDA, "No CUDA GPU detected. This benchmark requires a GPU runtime."

try:
    import vllm; vllm_ver = vllm.__version__
except Exception:
    vllm_ver = "?"

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": torch.cuda.get_device_name(0),
    "gpu_count": torch.cuda.device_count(),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "cuda": torch.version.cuda,
    "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__,
    "vllm": vllm_ver,
    "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 3. Configuration
The small tier (≤16 GB) is the **apples-to-apples anchor** that runs on a T4
everywhere; the large tier only runs on bigger GPUs (your A100). Models are
ungated (Qwen2.5) so no HF token is needed.

In [ ]:
VRAM = ENV["vram_total_gb"]
TIER = "large" if VRAM >= 24 else "small"

MODEL = "Qwen/Qwen2.5-7B-Instruct" if TIER == "large" else "Qwen/Qwen2.5-1.5B-Instruct"

# vLLM's default `--dtype auto` resolves to bf16 for bf16 model configs (Qwen2.5),
# but pre-Ampere GPUs (e.g. Colab T4, compute capability 7.5) have no bf16 support
# and vLLM refuses to start. Force fp16 there so the advertised T4 path actually runs.
_cc_major = torch.cuda.get_device_capability(0)[0]
DTYPE = "auto" if _cc_major >= 8 else "half"

# Dataset: 'sharegpt' = community-comparable (realistic length distribution),
#          'random'   = fixed input/output lengths, no download, fully reproducible.
DATASET = "sharegpt"          # or "random"
RANDOM_INPUT_LEN = 1024        # used only when DATASET == "random"
RANDOM_OUTPUT_LEN = 1024

NUM_PROMPTS = 300              # requests per sweep point (raise on A100)
REQUEST_RATES = [4, 16, "inf"] # req/s; "inf" = fire all at once (max-throughput point)

PORT = 8000
GPU_MEM_UTIL = 0.90
MAX_MODEL_LEN = 4096
OUT_DIR = "vllm_bench_results"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"VRAM={VRAM} GB -> TIER={TIER}, MODEL={MODEL}, DATASET={DATASET}, DTYPE={DTYPE}")

## 4. Dataset
ShareGPT is the conversation distribution most public vLLM numbers use, so your
results line up with theirs. (Skip-downloads automatically for `random`.)

In [ ]:
import urllib.request

SHAREGPT_URL = ("https://huggingface.co/datasets/anon8231489123/"
                "ShareGPT_Vicuna_unfiltered/resolve/main/"
                "ShareGPT_V3_unfiltered_cleaned_split.json")
SHAREGPT_PATH = os.path.join(OUT_DIR, "ShareGPT_V3_unfiltered_cleaned_split.json")

if DATASET == "sharegpt" and not os.path.exists(SHAREGPT_PATH):
    print("Downloading ShareGPT (~200 MB, one time)...")
    urllib.request.urlretrieve(SHAREGPT_URL, SHAREGPT_PATH)
    print("Saved:", SHAREGPT_PATH)
else:
    print("Dataset ready (or using 'random').")


## 5. Launch the vLLM OpenAI server
We start a real OpenAI-compatible server in the background — exactly the
production serving path — then wait for `/health`.

In [ ]:
import subprocess, time, json, urllib.request

def health_ok(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def served_models(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=5) as r:
            return [m.get("id") for m in json.load(r).get("data", [])]
    except Exception:
        return []

# Refuse to start if something already owns the port, so the sweep can't be
# attributed to a stale server left over from a previous run.
if health_ok(PORT):
    raise RuntimeError(
        f"A server is already live on port {PORT}. Run the teardown cell (or "
        "restart the runtime) before launching, so results aren't recorded "
        "against a stale server.")

server_log = open(os.path.join(OUT_DIR, "server.log"), "w")
server_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", MODEL,
    "--port", str(PORT),
    "--dtype", DTYPE,
    "--gpu-memory-utilization", str(GPU_MEM_UTIL),
    "--max-model-len", str(MAX_MODEL_LEN),
    "--disable-log-requests",
]
print("Starting:", " ".join(server_cmd))
server = subprocess.Popen(server_cmd, stdout=server_log, stderr=subprocess.STDOUT)

def wait_health(port, timeout=900):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if server.poll() is not None:
            raise RuntimeError("Server exited early — check vllm_bench_results/server.log")
        if health_ok(port):
            # Confirm the live server is the one we just started, not a stale
            # process bound to the same port serving a different model.
            models = served_models(port)
            if MODEL not in models:
                raise RuntimeError(
                    f"/health is up but /v1/models reports {models}, not {MODEL!r} — "
                    "a stale server likely owns the port; run teardown or restart.")
            print(f"Server healthy after {int(time.time()-t0)}s, serving {MODEL}")
            return True
        time.sleep(5)
    raise TimeoutError("Server did not become healthy in time.")

wait_health(PORT)

## 6. Run the load generator (request-rate sweep)
For each request rate we run vLLM's `bench serve` and collect the standardized
metrics JSON. The sweep traces the latency-vs-throughput curve.

In [ ]:
import glob, json

def run_bench(rate):
    tag = f"rate_{rate}"
    res_file = os.path.join(OUT_DIR, f"bench_{tag}.json")
    # Drop any result from a previous run so a failed sweep can't silently
    # load stale metrics for the current model/platform/config.
    if os.path.exists(res_file):
        os.remove(res_file)
    cmd = [
        sys.executable, "-m", "vllm.entrypoints.cli.main", "bench", "serve",
        "--backend", "vllm",
        "--model", MODEL,
        "--host", "127.0.0.1", "--port", str(PORT),
        "--num-prompts", str(NUM_PROMPTS),
        "--request-rate", str(rate),
        "--save-result", "--result-filename", res_file,
    ]
    if DATASET == "sharegpt":
        cmd += ["--dataset-name", "sharegpt", "--dataset-path", SHAREGPT_PATH]
    else:
        cmd += ["--dataset-name", "random",
                "--random-input-len", str(RANDOM_INPUT_LEN),
                "--random-output-len", str(RANDOM_OUTPUT_LEN)]
    print(f"\n=== request-rate={rate} ===")
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1500:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-1500:])
        return {"error": f"bench exited {p.returncode}", "stderr_tail": p.stderr[-500:]}
    if os.path.exists(res_file):
        with open(res_file) as f:
            return json.load(f)
    return {"error": "no result file", "stderr_tail": p.stderr[-500:]}

SWEEP = {}
for rate in REQUEST_RATES:
    SWEEP[str(rate)] = run_bench(rate)

## 7. Results — normalize, save, summarize

In [ ]:
import pandas as pd

KEYS = ["request_throughput", "output_throughput", "total_token_throughput",
        "mean_ttft_ms", "median_ttft_ms", "p99_ttft_ms",
        "mean_tpot_ms", "median_tpot_ms", "p99_tpot_ms",
        "mean_itl_ms", "median_itl_ms"]

rows = []
for rate, res in SWEEP.items():
    row = {"request_rate": rate}
    for k in KEYS:
        row[k] = res.get(k) if isinstance(res, dict) else None
    rows.append(row)
df = pd.DataFrame(rows)

NORM = {
    "schema": "vllm-serving-bench/1.0",
    "env": ENV, "tier": TIER, "model": MODEL,
    "dataset": DATASET, "num_prompts": NUM_PROMPTS,
    "request_rates": [str(r) for r in REQUEST_RATES],
    "sweep": SWEEP,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"]).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"vllm_serving_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)

print("Saved:", out)
print("\n==== SERVING SUMMARY ====")
print(f"{ENV['gpu_name']} ({ENV['vram_total_gb']} GB) | {MODEL} | {DATASET}")
try:
    from IPython.display import display
    display(df)
except Exception:
    print(df.to_string(index=False))


## 8. Teardown
Stop the server to free the GPU before running anything else.

In [ ]:
try:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        # Graceful shutdown stalled (e.g. unloading a large model); force-kill so
        # the process can't linger on the port/GPU and break the next run.
        print("Graceful stop timed out; force-killing.")
        server.kill()
        server.wait(timeout=30)
    print("Server stopped.")
except Exception as e:
    print("Stop error:", e)
finally:
    try: server_log.close()
    except Exception: pass


## 9. What makes this comparable (and what to do next)

**Comparable because:** optimized runtime (vLLM, PagedAttention + continuous
batching) measures the *hardware ceiling*, not eager-mode overhead; a real
**ShareGPT** request distribution is what public numbers use; and a standardized
**load generator** reports the same TTFT / TPOT / throughput percentiles
everyone reports.

**To read the result:** find the request rate where p99 TTFT crosses your SLA —
that's your max sustainable load (capacity). The `"inf"` row is the raw
max-throughput ceiling (offline-style).

**Next rungs of credibility:**
- **MLPerf Inference** (`language/llama3.1-8b`, Server + Offline scenarios) for
  leaderboard-grade, *accuracy-gated* numbers.
- **TensorRT-LLM** for the peak A100 ceiling (often big TTFT wins vs vLLM).
- **optimum-benchmark** to compare PyTorch vs ONNX Runtime vs TensorRT on the
  same model with one harness.

See `BENCHMARKING_STRATEGY.md` for the full plan and run matrix.
